# 03 Cleaning

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic_raw.csv')
print(f"Starting shape: {df.shape}")

# --- Step 1: Handle duplicate check (none found, but verify programmatically) ---
n_dupes = df.duplicated(subset=[c for c in df.columns if c != 'PassengerId']).sum()
print(f"Duplicate rows (excluding ID): {n_dupes}")

# --- Step 2: Fix data types ---
df['Survived'] = df['Survived'].astype('category')
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')
print("Dtypes converted for categorical columns: Survived, Pclass, Sex")

# --- Step 3: Missing Embarked (2 records) — impute with mode ---
mode_embarked = df['Embarked'].mode()[0]
missing_embarked_ids = df.loc[df['Embarked'].isnull(), 'PassengerId'].tolist()
df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Embarked: filled {len(missing_embarked_ids)} missing values (PassengerId {missing_embarked_ids}) with mode '{mode_embarked}'")

# --- Step 4: Missing Age (177 records, ~19.9%) ---
# Strategy: median imputation grouped by Pclass + Sex + Title (extracted from Name),
# since age correlates strongly with passenger class and social title.
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare', 'the Countess': 'Rare'
}
df['Title'] = df['Title'].replace(title_map)
print("\nTitle extracted from Name; counts:")
print(df['Title'].value_counts())

age_before_missing = df['Age'].isnull().sum()
df['Age'] = df.groupby(['Pclass', 'Sex', 'Title'], observed=True)['Age']               .transform(lambda x: x.fillna(x.median()))
# Fallback for any remaining NaN (rare Title/Pclass/Sex combos with all-NaN age)
still_missing = df['Age'].isnull().sum()
if still_missing > 0:
    df['Age'] = df.groupby(['Pclass', 'Sex'], observed=True)['Age']                   .transform(lambda x: x.fillna(x.median()))
final_missing_age = df['Age'].isnull().sum()
print(f"\nAge: {age_before_missing} missing -> grouped-median imputation (Pclass+Sex+Title) "
      f"-> {still_missing} remained -> Pclass+Sex fallback -> {final_missing_age} missing")

# Age bucket / IsChild flag for downstream analysis (useful derived feature)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100],
                        labels=['Child','Teen','Young Adult','Adult','Senior'])

# --- Step 5: Cabin — 77% missing, too sparse to impute meaningfully ---
# Rather than dropping the column (loses signal), engineer a binary "HasCabin" indicator
# and extract the Deck letter where available.
df['HasCabin'] = df['Cabin'].notnull().astype(int)
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].fillna('Unknown')
print(f"\nCabin: 77.1% missing -> engineered HasCabin (binary) and Deck (first letter, 'Unknown' if missing)")
print(df['Deck'].value_counts())

# --- Step 6: Outliers in Fare ---
# Investigate zero-fare records: these correspond to crew/employees (e.g. Thomas Andrews,
# the Titanic's builder) traveling on employee passage, not data entry errors.
zero_fare = df[df['Fare'] == 0]
print(f"\nZero-fare passengers: {len(zero_fare)} — verified as legitimate (staff/guaranteed passage),"
      " retained but flagged")
df['FareIsZero'] = (df['Fare'] == 0).astype(int)

# High-fare outliers (IQR method) — legitimate first-class/group fares, not errors.
# We do NOT delete outliers (they are real, valid extreme values), but we create a
# winsorized/capped fare feature and a log-transformed feature for modeling stability.
q1, q3 = df['Fare'].quantile([.25, .75])
iqr = q3 - q1
upper_bound = q3 + 1.5*iqr
n_outliers = (df['Fare'] > upper_bound).sum()
print(f"Fare: {n_outliers} values exceed IQR upper bound ({upper_bound:.2f}); "
      f"retained as valid, added FareCapped (winsorized) and FareLog (log1p) features")
df['FareCapped'] = df['Fare'].clip(upper=upper_bound)
df['FareLog'] = np.log1p(df['Fare'])

# --- Step 7: Consistency checks ---
# SibSp/Parch sanity: family size feature, check for implausible values
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
implausible_family = df[df['FamilySize'] > 8]
print(f"\nFamilySize derived (SibSp+Parch+1). Records with FamilySize>8 (plausibility check): {len(implausible_family)}")

# Name/Sex consistency: verify title aligns with sex (e.g., no 'Mr.' tagged female)
mismatch = df[((df['Title'].isin(['Mr','Master'])) & (df['Sex']=='female')) |
              ((df['Title'].isin(['Mrs','Miss'])) & (df['Sex']=='male'))]
print(f"Title/Sex mismatches found: {len(mismatch)}")

# --- Step 8: Drop columns not useful for downstream modeling ---
# Ticket is a high-cardinality free-text identifier with no consistent structure; Cabin
# retained info already captured via HasCabin/Deck, so raw Cabin can be dropped for the
# modeling-ready dataset (kept in the "cleaned but full" version).
df_cleaned_full = df.copy()
df_model_ready = df.drop(columns=['Cabin', 'Ticket', 'Name']).copy()

print(f"\nFinal cleaned shape (full): {df_cleaned_full.shape}")
print(f"Final model-ready shape: {df_model_ready.shape}")
print(f"\nRemaining missing values (full):\n{df_cleaned_full.isnull().sum()[df_cleaned_full.isnull().sum()>0]}")

df_cleaned_full.to_csv('titanic_cleaned_full.csv', index=False)
df_model_ready.to_csv('titanic_model_ready.csv', index=False)

print("\nSaved: titanic_cleaned_full.csv, titanic_model_ready.csv")
